In [139]:
import os
import pandas as pd
from scipy.stats import chi2_contingency
from statsmodels.stats.contingency_tables import Table
import math
from collections import defaultdict
from itertools import product
from typing import Tuple, Optional

In [140]:
sp18_ext_path = "../resources/developed/sp18/external_effect"

res_outdir = "../out/external_effect"

incsv_args = {
	"sep":"\t",
    "header":0,
    "index_col":"const_comb",
    # all str, WC int, is linker pluralic bool
    "dtype": defaultdict(
        lambda: str,
        wc="int",
        is_linker_pl="bool"
    ),
	# = csv.QUOTE_NONE, separate by tabs only
    "quoting": 3
}

outcsv_args = {
    "sep": "\t",
    "index": True,
    "header": True
}

In [141]:
def calc_pll_shares(
    sp18_sample: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    # calculate absolute counts and shares of compounds 
    # with pluralic linkers in Sg vs. Pl forms

    sp18_sample_pll_counts_data = {}
    for const_comb, group in sp18_sample.groupby("const_comb"):
        for num in ["sg", "pl"]:
            sp18_sample_pll_counts_data[(const_comb, f"num_{num}")] = {
                "l_sg": len(
                    group[(group["n2_num"] == num) &
                    (~group["is_linker_pl"])]
                ),
                "l_pl": len(
                    group[(group["n2_num"] == num) &
                    (group["is_linker_pl"])]
                )
            }

    sp18_sample_pllcounts = pd.DataFrame(sp18_sample_pll_counts_data).T
    sp18_sample_pllcounts["pl_share"] = sp18_sample_pllcounts["l_pl"] / (
        sp18_sample_pllcounts["l_sg"] + sp18_sample_pllcounts["l_pl"]
    )

    sp18_sample_pllshares = sp18_sample_pllcounts \
        .unstack()["pl_share"]  \
        [["num_sg", "num_pl"]]

    return sp18_sample_pllcounts, sp18_sample_pllshares

In [142]:
def sp18_orig_analysis(
    sp18_sample_pllcounts: pd.DataFrame
) -> Tuple[pd.Series, pd.Series, pd.Series]:

    # Methodology from the original [schaefer_pankratz_2018:26] paper:
    # 1. Create contingency table:
    # n2_num (Sg vs. Pl) x n1_pl_marker (zero vs. pluralic linker).
    # 2. Perform the chi-squared test of independence over the table
    # to assess the statistical significance of the association between
    # the Pl form of N2 and the presence of a pluralic linker in N1.
    # 3. Perform the Cramer's V test to assess the strength
    # of the association.
    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # 1. Create contingency table replicating [schaefer_pankratz_2018:26]
    contingency_table = sp18_sample_pllcounts \
        .groupby(level=1)[["l_sg", "l_pl"]].sum()   \
        .T  \
        # [["num_sg", "num_pl"]]

    # 2. Perform the chi-squared test.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `chi2_contingency` function from `scipy.stats`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    chi2_res = chi2_contingency(contingency_table)

    # 3. Perform the Cramer's V test to assess 
    # the strength of the association.

    # the original R code for Cramer's V is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   cramer.v <- function(chisq) {
    #       unname(sqrt(chisq$statistic/(sum(chisq$observed)*min(dim(chisq$observed)-1))))
    #   }
    #
    # we will reconstruct it in Python:
    #   dim(chisq$observed) -> ic_alt_nipe_sg_l_ct.shape
    #
    #   min(dim(chisq$observed)-1) -> min(
    #       ic_alt_nipe_sg_l_ct.shape[0] - 1,
    #       ic_alt_nipe_sg_l_ct.shape[1] - 1
    #   )
    # 
    #   sum(chisq$observed) -> ic_alt_nipe_sg_l_ct.values.sum()
    #
    #   unname -> [irrelevant for Python]
    def cramer_v() -> float:
        return math.sqrt(
            chi2_res.statistic / (
                contingency_table.values.sum() *
                min(
                    contingency_table.shape[0] - 1,
                    contingency_table.shape[1] - 1
                )
            )
        )

    cramer_v_val = cramer_v()

    # the original R code for Cohen's W is available in
    # linkingelements/Data/R/functions.R, and is as follows:
    # 
    #   # cohen.w <- function(chisq) {
    #       cramer.v(chisq) * sqrt(min(dim(chisq$observed)-1))
    #   }
    def cohen_w() -> float:
        return cramer_v_val * math.sqrt(
            min(
                contingency_table.shape[0] - 1,
                contingency_table.shape[1] - 1
            )
        )

    cohen_w_val = cohen_w()

    # 4. Inspect symmetrical standardized residuals to identify
    # the cells that contribute to the association, and in which direction.

    # even though I didn't find the original R code
    # of [schaefer_pankratz_2018] for this experiment,
    # the `standardized_resids` property of
    # `statsmodels.stats.contingency_tables.Table`
    # produced the same results as in the paper
    # on the same contingency table, so we can be
    # confident that we are replicating the same analysis
    resids = Table(contingency_table)    \
        .standardized_resids.round(3)
    
    chi2df = pd.Series(
        data={
            "N": contingency_table.values.sum(),
            "df": chi2_res.dof,
            "chi2": chi2_res.statistic.round(3),
            "p": chi2_res.pvalue,#.round(3),
            "cramer_v": round(cramer_v_val, 3),
            "cohen_w": round(cohen_w_val, 3),
            "resids_cong": resids.loc["l_pl", "num_pl"].item(),
            "resids_incong": resids.loc["l_pl", "num_sg"].item()
        }
    )

    return contingency_table, chi2df

In [143]:
def run_study(
	sp18_sample: pd.DataFrame,
	outdir: Optional[str]=None
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

	# calculate absolute counts and shares of compounds 
	# with pluralic linkers in Sg vs. Pl forms
	sp18_sample_pllcounts, sp18_sample_pllshares = calc_pll_shares(
		sp18_sample
	)

	# shares desc
	sp18_sample_pllshares_desc = sp18_sample_pllshares.describe()

	# perform the original analysis from [schaefer_pankratz_2018:26]
	ct, chi2df = sp18_orig_analysis(sp18_sample_pllcounts)

	if outdir:

		os.makedirs(outdir, exist_ok=True)

		sp18_sample_pllcounts.to_csv(
			os.path.join(outdir, "pllcounts.tsv"),
			**outcsv_args
		)
		sp18_sample_pllshares.to_csv(
			os.path.join(outdir, "pllshares.tsv"),
			**outcsv_args
		)

		sp18_sample_pllshares_desc.to_csv(
			os.path.join(outdir, "pllshares_desc.tsv"),
			**outcsv_args
		)

		ct.to_csv(
			os.path.join(outdir, "contingency_table.tsv"),
			**outcsv_args
		)
		chi2df.to_csv(
			os.path.join(outdir, "chi2.tsv"),
			**{**outcsv_args, "header": False}
		)

	return (
		sp18_sample_pllcounts, sp18_sample_pllshares, 
		sp18_sample_pllshares_desc,
		ct, chi2df
	)

# Study 1

Compounds built from non-unique constituent combinations (regardless of the linker) appearing both in the Sg and the Pl forms and not undergoing the IntPE whose N1 may not attach a pluralic linker.

In [144]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing both in the Sg and the Pl forms
# and not undergoing the IntPE whose N1 may not attach a pluralic linker
# (N=1236, UniqN=665, UniqCN=468, UniqN1N=30)
sp18_s1 = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_nipe_npll.tsv"),
    **incsv_args
)

*_, chi2df_s1 = run_study(sp18_s1, os.path.join(res_outdir, "ic_alt_nipe_npll"))

In [145]:
chi2df_s1.round(3)

N                1236.000
df                  1.000
chi2                1.722
p                   0.189
cramer_v            0.037
cohen_w             0.037
resids_cong         1.370
resids_incong      -1.370
dtype: float64

In [146]:
# not low freq
sp18_s1_nlf = sp18_s1[sp18_s1["wc"] >= 20]

*_, chi2df_s1_nlf = run_study(sp18_s1_nlf, os.path.join(res_outdir, "ic_alt_nipe_npll/nlf"))

In [147]:
chi2df_s1_nlf.round(3)

N                808.000
df                 1.000
chi2               0.049
p                  0.824
cramer_v           0.008
cohen_w            0.008
resids_cong        0.296
resids_incong     -0.296
dtype: float64

In [148]:
# high freq
sp18_s1_hf = sp18_s1[sp18_s1["wc"] >= 100]

*_, chi2df_s1_hf = run_study(sp18_s1_hf, os.path.join(res_outdir, "ic_alt_nipe_npll/hf"))

In [149]:
chi2df_s1_hf.round(3)

N                694.000
df                 1.000
chi2               0.026
p                  0.872
cramer_v           0.006
cohen_w            0.006
resids_cong       -0.242
resids_incong      0.242
dtype: float64

# Study 2

Compounds built from non-unique constituent combinations (regardless of the linker) appearing both in the Sg and the Pl forms and not undergoing the IntPE.

In [150]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing both in the Sg and the Pl forms
# and not undergoing the IntPE
# (N=1931, UniqN=1043, UniqCN=724)
sp18_s2 = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_nipe.tsv"),
    **incsv_args
)

*_, chi2df_s2 = run_study(sp18_s2, os.path.join(res_outdir, "ic_alt_nipe"))

In [151]:
chi2df_s2.round(3)

N                1931.000
df                  1.000
chi2                1.994
p                   0.158
cramer_v            0.032
cohen_w             0.032
resids_cong         1.458
resids_incong      -1.458
dtype: float64

In [152]:
# not low freq
sp18_s2_nlf = sp18_s2[sp18_s2["wc"] >= 20]

*_, chi2df_s2_nlf = run_study(sp18_s2_nlf, os.path.join(res_outdir, "ic_alt_nipe/nlf"))

In [153]:
chi2df_s2_nlf.round(3)

N                1248.000
df                  1.000
chi2                0.139
p                   0.710
cramer_v            0.011
cohen_w             0.011
resids_cong         0.430
resids_incong      -0.430
dtype: float64

In [154]:
# high freq
sp18_s2_hf = sp18_s2[sp18_s2["wc"] >= 100]

*_, chi2df_s2_hf = run_study(sp18_s2_hf, os.path.join(res_outdir, "ic_alt_nipe/hf"))

In [155]:
chi2df_s2_hf.round(3)

N                1076.000
df                  1.000
chi2                0.000
p                   1.000
cramer_v            0.000
cohen_w             0.000
resids_cong         0.028
resids_incong      -0.028
dtype: float64

# Study 3

Compounds built from non-unique constituent combinations (regardless of the linker) appearing both in the Sg and the Pl forms.

In [156]:
# Compounds built from non-unique constituent combinations (regardless 
# of the linker) appearing both in the Sg and the Pl forms
# (N=1931, UniqN=1043, UniqCN=724)
sp18_s3 = pd.read_csv(
    os.path.join(sp18_ext_path, "ic_alt_all.tsv"),
    **incsv_args
)

*_, chi2df_s3 = run_study(sp18_s3, os.path.join(res_outdir, "ic_alt_all"))

In [157]:
chi2df_s3.round(3)

N                2318.000
df                  1.000
chi2                1.828
p                   0.176
cramer_v            0.028
cohen_w             0.028
resids_cong         1.394
resids_incong      -1.394
dtype: float64

In [158]:
# not low freq
sp18_s3_nlf = sp18_s3[sp18_s3["wc"] >= 20]

*_, chi2df_s3_nlf = run_study(sp18_s3_nlf, os.path.join(res_outdir, "ic_alt_all/nlf"))

In [159]:
chi2df_s3_nlf.round(3)

N                1516.000
df                  1.000
chi2                0.175
p                   0.676
cramer_v            0.011
cohen_w             0.011
resids_cong         0.470
resids_incong      -0.470
dtype: float64

In [160]:
# high freq
sp18_s3_hf = sp18_s3[sp18_s3["wc"] >= 100]

*_, chi2df_s3_hf = run_study(sp18_s3_hf, os.path.join(res_outdir, "ic_alt_all/hf"))

In [161]:
chi2df_s3_hf.round(3)

N                1308.000
df                  1.000
chi2                0.012
p                   0.914
cramer_v            0.003
cohen_w             0.003
resids_cong         0.163
resids_incong      -0.163
dtype: float64

# Study 4

Individual N1s.

In [162]:
n1s = sp18_s3["n1"].unique()
n1s

<StringArray>
[      'apfel',         'bad',        'ball',        'bett',        'bild',
       'brett',       'brief',        'buch',       'bucht',       'dämon',
          'ei', 'eigenschaft',     'element',        'frau',       'gerät',
    'geräusch',    'geschenk',     'gitarre',        'hand',        'haus',
        'hemd',        'hund',       'katze',        'kind',     'kommune',
        'lied',        'loch',      'mutter',       'nagel',      'nation',
         'ohr',      'person',     'produkt',   'projektor',         'rad',
     'schloss',     'schwert',   'schwester',       'sonne',       'stadt',
       'vater',       'vogel',         'weg',        'wurm',        'zahn',
       'zitat']
Length: 46, dtype: str

In [ ]:
for n1, (sp18, sp18_nlf, sp18_hf, outdir_prfx) in product(
	n1s,
	[
		(sp18_s1, sp18_s1_nlf, sp18_s1_hf, "ic_alt_nipe_npll/n1"),
		(sp18_s2, sp18_s2_nlf, sp18_s2_hf, "ic_alt_nipe/n1"),
		(sp18_s3, sp18_s3_nlf, sp18_s3_hf, "ic_alt_all/n1")
	]
):
	
	for sp18, freq_cl in [
		(sp18, ""),
		(sp18_nlf, "nlf"),
		(sp18_hf, "hf")
	]:
		
		outdir = os.path.join(res_outdir, outdir_prfx, freq_cl)
		os.makedirs(outdir, exist_ok=True)

		n1_group = sp18[sp18["n1"] == n1]
		if n1_group.empty:
			continue
		try:
			*_, chi2df = run_study(n1_group)
			# increased threshold for significance to 0.1 
			# to account for the small sample size;
			# the eventual verdict will be based on 
			# the Cramer's V and residuals, not the p-value
			if chi2df["p"] < 0.1:
				chi2df.to_csv(
					os.path.join(outdir, f"{n1}.tsv"),
					**{**outcsv_args, "header": False}
				)
		except ValueError:
			# zeros in contingency table, cannot compute chi2
			pass

In [170]:
# [pankratz_2017]'s N1:
pankratz_n1s = [
	"geschenk",
	"apfel",
	"product",
	"stadt",
	"bad",
	"ball",
	"person",
	"vater"
]

for n1, (sp18, outdir_prfx) in product(
	pankratz_n1s, 
	[
		(sp18_s1, "ic_alt_nipe_npll/n1"),
		(sp18_s2, "ic_alt_nipe/n1"),
		(sp18_s3, "ic_alt_all/n1")
	]
):
	
	outdir = os.path.join(res_outdir, outdir_prfx, "pankratz_2017")
	os.makedirs(outdir, exist_ok=True)

	n1_group = sp18[sp18["n1"] == n1]
	if n1_group.empty:
		continue
	try:
		*_, chi2df = run_study(n1_group)
		# increased threshold for significance to 0.1 
		# to account for the small sample size;
		# the eventual verdict will be based on 
		# the Cramer's V and residuals, not the p-value
		chi2df.to_csv(
			os.path.join(outdir, f"{n1}.tsv"),
			**{**outcsv_args, "header": False}
		)
	except ValueError:
		# zeros in contingency table, cannot compute chi2
		pass